# GEPA Prompt Optimization — Skill Builder Pipeline

**Goal:** Optimize the 3 agent prompts (Planner, Curator, Generator) simultaneously using DSPy's GEPA optimizer, with GPT-4.1 as the reflection model and an LLM-as-judge metric.

**How it works:**
1. Run the 3-agent pipeline on skill requests from `skill_requests.csv`
2. An LLM judge (GPT-4.1) scores each generated `skill.md` across 5 dimensions
3. GEPA's reflection model reads execution traces + judge feedback, proposes targeted prompt edits
4. Pareto frontier preserves diverse best candidates; crossover merges strengths
5. After convergence, export optimized prompts

**Why this matters:** One GEPA run optimizes 3 prompts → benefits *all future skills* the builder generates.


In [6]:
import sys
!{sys.executable} -m pip install dspy

  Using cached dspy-3.2.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached json_repair-0.59.10-py3-none-any.whl.metadata (19 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached asyncer-0.0.8-py3-none-any.whl.metadata (6.7 kB)
  Using cached gepa-0.0.27-py3-none-any.whl.metadata (30 kB)
  Using cached typeguard-4.4.3-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached aiohappyeyeballs-2.6.2-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Users\\pranavbe\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\jsonschema\\__init__.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -

  Using cached dspy-3.2.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached openai-2.38.0-py3-none-any.whl.metadata (31 kB)
  Using cached litellm-1.86.0-py3-none-any.whl.metadata (33 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached aiohttp-3.13.5-cp312-cp312-win_amd64.whl.metadata (8.4 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached huggingface_hub-1.16.1-py3-none-any.whl.metadata (14 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
Using cached dspy-3.2.1-py3-none-any.whl (331 kB)
Using cached litellm-1.86.0-py3-none-any.whl (17.0 MB)
Using cached aiohttp-3.13.5-cp312-cp312-win_amd64.whl (460 kB)
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached openai-2.38.0-py3-none-any.whl (1.3 MB)
Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl (2.8 MB)
Using cached huggingface_hub-1.16.

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
import csv
import json
import os
import re
from pathlib import Path
from typing import Any, Dict, List, Optional

import dspy
import mlflow
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is missing. Add it to .env or set it in the terminal.")

print("DSPy version:", dspy.__version__)
print("MLflow version:", mlflow.__version__)
print("Pandas version:", pd.__version__)


DSPy version: 3.2.1
MLflow version: 3.9.0
Pandas version: 2.3.3


In [38]:
# ── Model Configuration ──

TASK_MODEL = "openai/gpt-4o-mini"
REFLECTION_MODEL = "openai/gpt-4.1"
JUDGE_MODEL = "gpt-4.1"  # direct OpenAI API call, not DSPy

# Task LM — used by the pipeline agents
task_lm = dspy.LM(model=TASK_MODEL, max_tokens=2000)
dspy.configure(lm=task_lm)

# Reflection LM — used by GEPA for meta-cognitive trace analysis
reflection_lm = dspy.LM(model=REFLECTION_MODEL, temperature=1.0, max_tokens=32000)

# Judge client — direct OpenAI client for LLM-as-judge evaluation
judge_client = OpenAI()

print(f"Task LM: {TASK_MODEL}")
print(f"Reflection LM: {REFLECTION_MODEL}")
print(f"Judge model: {JUDGE_MODEL}")


Task LM: openai/gpt-4o-mini
Reflection LM: openai/gpt-4.1
Judge model: gpt-4.1


In [ ]:
# ── MLflow Setup ──

from datetime import datetime

mlflow.set_tracking_uri("http://172.27.72.27:5000")
mlflow.dspy.autolog()

SESSION_ID = datetime.now().strftime("%Y%m%d_%H%M")

def start_phase_run(phase: str):
    """Start a parent MLflow run for a named phase with standard tags."""
    run = mlflow.start_run(run_name=f"{phase}_{SESSION_ID}")
    mlflow.set_tags({
        "app": "skill_builder",
        "phase": phase,
        "pipeline_version": "1.1.0",
        "signature_version": "2.0.0",
        "dataset_version": "skill_requests_v1",
        "session_id": SESSION_ID,
    })
    return run

print(f"MLflow configured. Session: {SESSION_ID}")


## Load & Split Skill Requests

Load the 10 skill requests and split into 7 training and 3 validation.


In [ ]:
def load_skill_requests(csv_path: str = "skill_requests.csv") -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["id"] = df["id"].astype(int)
    return df

all_requests = load_skill_requests()
print(f"Loaded {len(all_requests)} skill requests")
display(all_requests)




Loaded 10 skill requests


,id,request
0,1,Create a skill for a data analyst that can cle...
1,2,Create a skill for a Python developer that can...
2,3,Create a skill for an ML engineer that can eva...
3,4,Create a skill for a data engineer that can va...
4,5,Create a skill for a business analyst that can...
5,6,Create a skill for a DevOps engineer that can ...
6,7,Create a skill for an AI engineer that can tra...
7,8,Create a skill for a product manager that can ...
8,9,Create a skill for a SQL analyst that can impr...
9,10,Create a skill for a documentation writer that...


In [41]:
# ── Train / Validation Split ──

SEED = 42
VAL_SIZE = 3

# Shuffle deterministically, take first 7 for train, last 3 for val
train_df = all_requests.sample(frac=1, random_state=SEED).iloc[:-VAL_SIZE].sort_index()
val_df = all_requests.sample(frac=1, random_state=SEED).iloc[-VAL_SIZE:].sort_index()

print("═══ Training set (7 requests) ═══")
display(train_df[["id", "request"]])
print()
print("═══ Validation set (3 requests) ═══")
display(val_df[["id", "request"]])


═══ Training set (7 requests) ═══


,id,request
0,1,Create a skill for a data analyst that can cle...
1,2,Create a skill for a Python developer that can...
2,3,Create a skill for an ML engineer that can eva...
5,6,Create a skill for a DevOps engineer that can ...
7,8,Create a skill for a product manager that can ...
8,9,Create a skill for a SQL analyst that can impr...
9,10,Create a skill for a documentation writer that...



═══ Validation set (3 requests) ═══


,id,request
3,4,Create a skill for a data engineer that can va...
4,5,Create a skill for a business analyst that can...
6,7,Create a skill for an AI engineer that can tra...


## Define the 3-Agent Pipeline

Same architecture as `agent_skillbuilder.py` — Planner → Curator → Generator.

```
User Request → PlannerAgent → CurationAgent → SkillGeneratorAgent → skill.md files
```


In [42]:
# ── DSPy Signatures ──

class PlannerClarificationSignature(dspy.Signature):
    """Break a raw skill request into a clear build plan.

    Identify every skill that must be created, preserve the user's requirements,
    and make the next agent's job easy. Do not write the final skill.md files.
    """

    user_request: str = dspy.InputField(desc="Raw user request or skill description")
    skill_inventory: str = dspy.OutputField(desc="Numbered list of skills to create, with each skill's purpose")
    clarified_plan: str = dspy.OutputField(desc="Step-by-step plan for creating the requested skill.md files")
    assumptions: str = dspy.OutputField(desc="Small assumptions made when the request is incomplete")
    curation_instructions: str = dspy.OutputField(desc="Instructions for the curation agent about what details to enrich")


class CurationEnrichmentSignature(dspy.Signature):
    """Convert the plan into concrete skill specifications.

    Add practical details, expected inputs/outputs, examples, and quality checks.
    Keep the content focused on what the generator needs to write good skill.md files.
    """

    skill_inventory: str = dspy.InputField(desc="Skills identified by the planner")
    clarified_plan: str = dspy.InputField(desc="Plan from the planner agent")
    curation_instructions: str = dspy.InputField(desc="Instructions from the planner for enrichment")
    skill_specs: str = dspy.OutputField(desc="Detailed specification for each skill: goal, inputs, outputs, workflow")
    examples: str = dspy.OutputField(desc="Concrete examples each skill should include or support")
    quality_checks: str = dspy.OutputField(desc="Checklist for validating the generated skill.md files")
    missing_info: str = dspy.OutputField(desc="Remaining gaps that need user input")


class SkillGenerationSignature(dspy.Signature):
    """Generate final skill.md files from curated specifications.

    Each file must be complete, readable, and include YAML frontmatter with
    name, description, inputs, and outputs.
    """

    skill_specs: str = dspy.InputField(desc="Detailed specification for each skill")
    examples: str = dspy.InputField(desc="Examples to include or support")
    quality_checks: str = dspy.InputField(desc="Checklist the generated files should satisfy")
    skill_markdown_files: List[str] = dspy.OutputField(desc="Complete markdown file contents, one string per skill")
    file_names: List[str] = dspy.OutputField(desc="Safe snake_case markdown filenames, one per skill")

In [43]:
# ── Agents ──

class PlannerAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_plan = dspy.ChainOfThought(PlannerClarificationSignature)

    def forward(self, user_request: str) -> Dict[str, str]:
        pred = self.generate_plan(user_request=user_request)
        return {
            "skill_inventory": pred.skill_inventory,
            "clarified_plan": pred.clarified_plan,
            "assumptions": pred.assumptions,
            "curation_instructions": pred.curation_instructions,
        }


class CurationAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.curate = dspy.ChainOfThought(CurationEnrichmentSignature)

    def forward(self, skill_inventory: str, clarified_plan: str, curation_instructions: str) -> Dict[str, str]:
        pred = self.curate(
            skill_inventory=skill_inventory,
            clarified_plan=clarified_plan,
            curation_instructions=curation_instructions,
        )
        return {
            "skill_specs": pred.skill_specs,
            "examples": pred.examples,
            "quality_checks": pred.quality_checks,
            "missing_info": pred.missing_info,
        }


class SkillGeneratorAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_skills = dspy.ChainOfThought(SkillGenerationSignature)

    def forward(self, skill_specs: str, examples: str, quality_checks: str) -> Dict[str, Any]:
        pred = self.generate_skills(
            skill_specs=skill_specs,
            examples=examples,
            quality_checks=quality_checks,
        )
        return {
            "skill_markdown_files": pred.skill_markdown_files,
            "file_names": pred.file_names,
        }


class SkillBuilderPipeline(dspy.Module):
    def __init__(self):
        super().__init__()
        self.planner = PlannerAgent()
        self.curator = CurationAgent()
        self.generator = SkillGeneratorAgent()

    def forward(self, user_request: str) -> Dict[str, Any]:
        plan_result = self.planner(user_request=user_request)
        curation_result = self.curator(
            skill_inventory=plan_result["skill_inventory"],
            clarified_plan=plan_result["clarified_plan"],
            curation_instructions=plan_result["curation_instructions"],
        )
        generation_result = self.generator(
            skill_specs=curation_result["skill_specs"],
            examples=curation_result["examples"],
            quality_checks=curation_result["quality_checks"],
        )
        return {
            "plan": plan_result,
            "curation": curation_result,
            "generated_skills": generation_result,
        }

print("Pipeline defined.")


Pipeline defined.


## LLM-as-Judge Metric

Evaluate each generated `skill.md` across 5 dimensions using GPT-4.1 as a judge.

| Dimension | What it checks | Weight |
|---|---|---|
| **YAML frontmatter** | Valid YAML with name, description, inputs, outputs | 20% |
| **Body clarity** | Clear, well-structured markdown body | 20% |
| **Actionability** | Concrete, specific steps an AI agent can follow | 20% |
| **Domain relevance** | Addresses the specific role/domain in the request | 20% |
| **Completeness** | Covers all requirements from the original request | 20% |

Each dimension scored 1–5. Total normalized to 0–1.

The feedback text returned alongside the score is what GEPA's reflection model reads to understand *why* a skill failed and *how* to fix the prompts.


In [44]:
JUDGE_TEMPLATE = """You are evaluating a generated skill.md file for an AI agent. The original request was:

{user_request}

Generated skill.md file:
{skill_markdown}

Evaluate on these 5 dimensions. For each, score 1 (poor) to 5 (excellent):

1. **YAML frontmatter** (1-5): Does it have valid YAML with `name`, `description`, `inputs`, and `outputs` fields?
2. **Body clarity** (1-5): Is the markdown body clear, well-organized, well-structured, and easy to read?
3. **Actionability** (1-5): Can an AI agent concretely follow these steps? Are instructions specific and actionable?
4. **Domain relevance** (1-5): Does the skill address the specific role and domain mentioned in the request?
5. **Completeness** (1-5): Does it cover ALL requirements from the original request without omissions?

Respond with valid JSON only (no markdown fences, no extra text):
{{
  "yaml_quality": <1-5>,
  "body_clarity": <1-5>,
  "actionability": <1-5>,
  "domain_relevance": <1-5>,
  "completeness": <1-5>,
  "overall_feedback": "<2-3 sentence summary explaining what's good and what needs improvement>"
}}"""

print("Judge template defined.")

Judge template defined.


In [45]:
def call_judge(user_request: str, skill_markdown: str) -> tuple[float, str, dict]:
    """Call GPT-4.1 judge and return (score_0_to_1, feedback_text, dimensions_dict)."""

    prompt = JUDGE_TEMPLATE.format(
        user_request=user_request,
        skill_markdown=skill_markdown,
    )

    response = judge_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=500,
    )

    raw = response.choices[0].message.content.strip()
    # Remove markdown fences if present
    raw = re.sub(r'^```json\s*', '', raw)
    raw = re.sub(r'\s*```$', '', raw)

    result = json.loads(raw)

    dimensions = {
        "yaml_quality": int(result["yaml_quality"]),
        "body_clarity": int(result["body_clarity"]),
        "actionability": int(result["actionability"]),
        "domain_relevance": int(result["domain_relevance"]),
        "completeness": int(result["completeness"]),
    }

    total = sum(dimensions.values())
    score = total / 25.0  # normalize to 0-1

    feedback = (
        f"Score: {score:.2f}/1.00 | "
        f"YAML={dimensions['yaml_quality']}/5, "
        f"Clarity={dimensions['body_clarity']}/5, "
        f"Actionability={dimensions['actionability']}/5, "
        f"Domain={dimensions['domain_relevance']}/5, "
        f"Completeness={dimensions['completeness']}/5. "
        f"Feedback: {result['overall_feedback']}"
    )

    return score, feedback, dimensions

print("Judge function defined.")


Judge function defined.


In [46]:
# ── GEPA Metric Function ──

def gepa_metric_fn(
    gold: dspy.Example,
    pred: Any,
    trace: Any = None,
    pred_name: Optional[str] = None,
    pred_trace: Any = None,
):
    """GEPA metric compatible with both dspy.Evaluate and GEPA's predictor feedback.

    Returns float at module level (for dspy.Evaluate compatibility).
    Returns {'score': float, 'feedback': str} at predictor level (for GEPA reflection).
    """

    user_request = gold.user_request

    # Extract generated skills from the prediction
    if hasattr(pred, 'generated_skills'):
        skills = pred.generated_skills.get('skill_markdown_files', [])
    elif isinstance(pred, dict) and 'generated_skills' in pred:
        skills = pred['generated_skills'].get('skill_markdown_files', [])
    else:
        skills = []

    if not skills:
        feedback = "No skills were generated. Pipeline produced empty output."
        if pred_name is not None:
            return {"score": 0.0, "feedback": feedback}
        return 0.0

    # Judge each generated skill and average the scores
    scores = []
    all_feedback = []

    for i, skill_md in enumerate(skills):
        score, feedback, dims = call_judge(user_request, skill_md)
        scores.append(score)
        all_feedback.append(f"Skill {i+1}: {feedback}")

    avg_score = sum(scores) / len(scores)
    combined_feedback = "\n".join(all_feedback)

    if pred_name is not None:
        # Predictor-level: return rich feedback for GEPA reflection
        combined_feedback = f"[{pred_name}] {combined_feedback}"
        return {"score": avg_score, "feedback": combined_feedback}
    else:
        # Module-level: return plain float for dspy.Evaluate compatibility
        return avg_score

print("GEPA metric function defined.")

GEPA metric function defined.


## Quick Test: Run Pipeline + Score with Judge

Verify the pipeline and metric work before starting the optimization run.


In [47]:
# Create a fresh pipeline
pipeline = SkillBuilderPipeline()

# Run on first training request
test_row = train_df.iloc[0]
test_request = test_row["request"]
test_id = test_row["id"]

print(f"Testing on request {test_id}...")
print(f"Request: {test_request[:120]}...")
print()

result = pipeline(user_request=test_request)

skills = result["generated_skills"]["skill_markdown_files"]
filenames = result["generated_skills"]["file_names"]

print(f"Generated {len(skills)} skill(s): {filenames}")
print()

# Display the first generated skill
from IPython.display import display, Markdown
if skills:
    display(Markdown("### Generated skill.md preview:"))
    display(Markdown(skills[0][:1500] + ("..." if len(skills[0]) > 1500 else "")))


Testing on request 1...
Request: Create a skill for a data analyst that can clean CSV files:
- remove duplicates
- handle missing values
- detect outlier...



2026/05/25 11:43:31 WARNING dspy.primitives.module: Failed to set LM usage. Please return `dspy.Prediction` object from dspy.Module to enable usage tracking.


Generated 3 skill(s): ['remove_duplicates.md', 'handle_missing_values.md', 'detect_outliers.md']



### Generated skill.md preview:

---
name: remove_duplicates
description: Ensure data integrity by eliminating duplicate rows in CSV files.
inputs:
  - CSV file with potentially duplicated rows
outputs:
  - Cleaned CSV file with duplicates removed
---

## Workflow
1. Read the CSV file.
2. Identify duplicate rows based on all or specific columns.
3. Remove duplicates from the dataset.
4. Save the cleaned data to a new CSV file.

## Example Implementation
```python
import pandas as pd

def remove_duplicates(file_path, output_path):
    # Read CSV file
    df = pd.read_csv(file_path)
    # Remove duplicate rows
    df_cleaned = df.drop_duplicates()
    # Save to new CSV file
    df_cleaned.to_csv(output_path, index=False)

# Example usage
remove_duplicates('data.csv', 'cleaned_data.csv')
```


Trace(trace_id=tr-1644164b64dbb6e2947a1809504333b9)

In [48]:
# Score with the judge
if skills:
    score, feedback, dims = call_judge(test_request, skills[0])

    print(f"Overall score: {score:.2f}")
    print(f"Feedback: {feedback}")
    print()

    # Show dimensions as a table
    dims_df = pd.DataFrame([dims], index=["Score"])
    dims_df["TOTAL"] = sum(dims.values())
    display(dims_df)
else:
    print("No skills to score.")


Overall score: 0.88
Feedback: Score: 0.88/1.00 | YAML=5/5, Clarity=5/5, Actionability=5/5, Domain=5/5, Completeness=2/5. Feedback: The skill.md file is well-structured, with clear YAML frontmatter and a detailed, actionable markdown body that is highly relevant to data analysts. However, it only covers the 'remove duplicates' aspect and omits handling missing values and detecting outliers, making it incomplete with respect to the original request. To fully meet requirements, additional skills for missing values and outlier detection should be included.



,yaml_quality,body_clarity,actionability,domain_relevance,completeness,TOTAL
Score,5,5,5,5,2,22


## Baseline Scores (Pre-Optimization)

Score all 7 training requests with the judge to establish a baseline.


In [ ]:
# ── Baseline Scores (Pre-Optimization) ──

baseline_scores = []

with start_phase_run("baseline"):
    mlflow.log_params({
        "task_model": TASK_MODEL,
        "judge_model": JUDGE_MODEL,
        "num_requests": len(train_df),
    })

    for _, row in train_df.iterrows():
        req_id = row["id"]
        req_text = row["request"]

        with mlflow.start_run(run_name=f"baseline_req_{req_id}", nested=True):
            mlflow.set_tags({
                "request_id": str(req_id),
                "phase": "baseline",
                "split": "train",
            })

            try:
                result = pipeline(user_request=req_text)
                skills = result["generated_skills"]["skill_markdown_files"]

                if skills:
                    score, feedback, dims = call_judge(req_text, skills[0])
                else:
                    score, feedback, dims = 0, "No skills generated", {}
            except Exception as e:
                score, feedback, dims = 0, f"Error: {e}", {}

            # Log judge scores as per-request metrics
            mlflow.log_metrics({
                "judge_score": score,
                "yaml_quality": dims.get("yaml_quality", 0),
                "body_clarity": dims.get("body_clarity", 0),
                "actionability": dims.get("actionability", 0),
                "domain_relevance": dims.get("domain_relevance", 0),
                "completeness": dims.get("completeness", 0),
            })

            baseline_scores.append({
                "request_id": req_id,
                "score": score,
                "feedback": feedback,
                **dims,
            })
            print(f"Request {req_id}: score={score:.2f}")

    baseline_df = pd.DataFrame(baseline_scores)
    avg_baseline = baseline_df["score"].mean()
    print(f"\nAverage baseline score: {avg_baseline:.2f}")
    print(f"Min: {baseline_df['score'].min():.2f}, Max: {baseline_df['score'].max():.2f}")

    # Log aggregate to parent run
    mlflow.log_metric("avg_baseline_score", avg_baseline)
    mlflow.log_metric("min_baseline_score", baseline_df["score"].min())
    mlflow.log_metric("max_baseline_score", baseline_df["score"].max())

display(baseline_df)


In [ ]:
# ── Save Baseline Generated Skills to baseline_skills/ ──

BASELINE_SKILLS_DIR = Path("baseline_skills")
BASELINE_SKILLS_DIR.mkdir(exist_ok=True)

print("Saving baseline (original) skills for all requests...")

for _, row in all_requests.iterrows():
    req_id = row["id"]
    req_text = row["request"]
    req_dir = BASELINE_SKILLS_DIR / f"request_{req_id}"
    req_dir.mkdir(exist_ok=True)

    try:
        result = pipeline(user_request=req_text)
        skills = result["generated_skills"]["skill_markdown_files"]
        filenames = result["generated_skills"]["file_names"]

        for content, fname in zip(skills, filenames):
            if not fname.endswith(".md"):
                fname += ".md"
            filepath = req_dir / fname
            filepath.write_text(content, encoding="utf-8")
            print(f"  [req {req_id}] Saved: {filepath}")

        # Also save the full pipeline trace as JSON
        trace_path = req_dir / "full_trace.json"
        trace_path.write_text(json.dumps(result, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

        # Log skills as MLflow artifacts
        mlflow.log_artifacts(str(req_dir), artifact_path=f"skills/baseline/request_{req_id}")

    except Exception as e:
        print(f"  [req {req_id}] ERROR: {e}")

print(f"\nBaseline skills saved to: {BASELINE_SKILLS_DIR.absolute()}")
print(f"Total request folders: {len(list(BASELINE_SKILLS_DIR.iterdir()))}")


## Run GEPA Optimization

Configure GEPA with:
- **Reflection model:** GPT-4.1 (reads traces + feedback → proposes prompt edits)
- **Budget:** `auto="medium"` (balanced optimization)
- **Candidate selection:** Pareto frontier (preserves diverse strengths)
- **Merge:** Enabled (crossover between successful lineages)




In [51]:
# ── Prepare Training Data ──

trainset = [
    dspy.Example(
        user_request=row["request"],
        request_id=row["id"],
    ).with_inputs("user_request")
    for _, row in train_df.iterrows()
]

print(f"Training set: {len(trainset)} examples")
print()

# Show the first example structure
print("Example structure:")
print(f"  inputs: {list(trainset[0].inputs().keys())}")
print(f"  user_request (first 100 chars): {trainset[0].user_request[:100]}...")


Training set: 7 examples

Example structure:
  inputs: ['user_request']
  user_request (first 100 chars): Create a skill for a data analyst that can clean CSV files:
- remove duplicates
- handle missing val...


In [ ]:
# ── Configure & Run GEPA ──

# Create a fresh pipeline for optimization
pipeline = SkillBuilderPipeline()

# Configure GEPA optimizer
gepa_optimizer = dspy.GEPA(
    metric=gepa_metric_fn,
    reflection_lm=reflection_lm,
    auto="light",
    candidate_selection_strategy="pareto",
    component_selector="round_robin",  # cycle through Planner → Curator → Generator
    use_merge=True,
    skip_perfect_score=True,
    num_threads=3,       # avoid parallel eval crashes with flaky API
    seed=42,
    log_dir="gepa_logs",
    track_stats=True,
)

print("GEPA optimizer configured.")
print(f"Auto budget: light")
print(f"Candidate selection: pareto")
print(f"Component selector: round_robin")
print(f"Merge: enabled")
print(f"Threads: 3 (parallel)")
print()

# Run optimization inside an MLflow parent run
with start_phase_run("optimization"):
    mlflow.log_params({
        "task_model": TASK_MODEL,
        "reflection_model": REFLECTION_MODEL,
        "judge_model": JUDGE_MODEL,
        "train_size": len(train_df),
        "val_size": len(val_df),
        "baseline_avg_score": avg_baseline,
        "gepa_auto": "light",
        "candidate_selection": "pareto",
        "component_selector": "round_robin",
        "use_merge": True,
        "skip_perfect_score": True,
        "num_threads": 3,
        "seed": 42,
    })

    optimized_pipeline = gepa_optimizer.compile(pipeline, trainset=trainset)

    # Log GEPA internal logs as artifacts
    if Path("gepa_logs").exists():
        mlflow.log_artifacts("gepa_logs", artifact_path="gepa_logs")

    print()
    print("Optimization complete!")


In [66]:
# ── Recover Best Pipeline From Checkpoint (if compile() crashed) ──
# Run this INSTEAD of the GEPA cell above. Loads the best candidate from gepa_logs.

import cloudpickle, numpy as np
from pathlib import Path

state_path = Path("gepa_logs") / "gepa_state.bin"
if state_path.exists():
    with open(state_path, "rb") as f:
        state = cloudpickle.load(f)

    candidates = state["program_candidates"]
    subscores = state["prog_candidate_val_subscores"]

    best_id, best_avg = max(
        ((i, np.mean(list(sd.values()))) for i, sd in enumerate(subscores) if sd),
        key=lambda x: x[1],
    )
    best_prog = candidates[best_id]

    print(f"Candidates: {len(candidates)}  |  Best: #{best_id}  |  Avg score: {best_avg:.4f}")
    print(f"Baseline: {avg_baseline:.4f}  |  Improvement: {(best_avg - avg_baseline) * 100:+.1f}%")

    pipeline_for_build = SkillBuilderPipeline()
    from dspy.teleprompt.gepa.gepa_utils import DspyAdapter
    adapter = DspyAdapter(
        student_module=pipeline_for_build,
        metric_fn=gepa_metric_fn,
        feedback_map={},
        failure_score=0.0,
    )
    optimized_pipeline = adapter.build_program(best_prog)
    print("optimized_pipeline ready. Now run Save Prompts and Save Skills cells.")
else:
    print("No gepa_state.bin found. Run the GEPA cell first.")


Candidates: 66  |  Best: #57  |  Avg score: 0.8781
Baseline: 0.8000  |  Improvement: +7.8%
optimized_pipeline ready. Now run Save Prompts and Save Skills cells.


In [67]:
adapter = DspyAdapter(
    student_module=pipeline_for_build,
    metric_fn=gepa_metric_fn,
    feedback_map={},
    failure_score=0.0,
)

## Save Optimized Prompts

Extract the optimized docstrings from the compiled pipeline's Signatures and save them.


In [68]:
OUTPUT_DIR = Path("optimized_prompts")
OUTPUT_DIR.mkdir(exist_ok=True)

# Access the optimized signatures through the compiled pipeline
# GEPA stores compiled programs with updated instructions
def extract_instruction(signature_cls) -> str:
    """Extract the instruction (docstring) from a DSPy Signature class."""
    return (signature_cls.__doc__ or "").strip()

# Try to get optimized instructions from the compiled pipeline's predictors
optimized_programs = {}

# The compiled pipeline has sub-modules. Access the ChainOfThought predictors.
if hasattr(optimized_pipeline, 'planner'):
    planner_cot = optimized_pipeline.planner.generate_plan
    if hasattr(planner_cot, 'extended_signature'):
        sig = planner_cot.extended_signature
        optimized_programs['planner'] = (sig.instructions or extract_instruction(PlannerClarificationSignature))

if hasattr(optimized_pipeline, 'curator'):
    curator_cot = optimized_pipeline.curator.curate
    if hasattr(curator_cot, 'extended_signature'):
        sig = curator_cot.extended_signature
        optimized_programs['curator'] = (sig.instructions or extract_instruction(CurationEnrichmentSignature))

if hasattr(optimized_pipeline, 'generator'):
    generator_cot = optimized_pipeline.generator.generate_skills
    if hasattr(generator_cot, 'extended_signature'):
        sig = generator_cot.extended_signature
        optimized_programs['generator'] = (sig.instructions or extract_instruction(SkillGenerationSignature))

# Save to files
for name, prompt_text in optimized_programs.items():
    fpath = OUTPUT_DIR / f"{name}_prompt.txt"
    fpath.write_text(prompt_text, encoding="utf-8")
    print(f"Saved: {fpath} ({len(prompt_text)} chars)")

# Also save baseline prompts for comparison
BASELINE_PROMPTS = {
    "planner": extract_instruction(PlannerClarificationSignature),
    "curator": extract_instruction(CurationEnrichmentSignature),
    "generator": extract_instruction(SkillGenerationSignature),
}

for name, prompt_text in BASELINE_PROMPTS.items():
    fpath = OUTPUT_DIR / f"{name}_prompt_baseline.txt"
    fpath.write_text(prompt_text, encoding="utf-8")
    print(f"Saved baseline: {fpath} ({len(prompt_text)} chars)")

print()
print("Optimized prompts saved to:", OUTPUT_DIR.absolute())


Saved baseline: optimized_prompts\planner_prompt_baseline.txt (211 chars)
Saved baseline: optimized_prompts\curator_prompt_baseline.txt (222 chars)
Saved baseline: optimized_prompts\generator_prompt_baseline.txt (179 chars)

Optimized prompts saved to: e:\GEPA\try2\optimized_prompts


In [ ]:
# ── Save Optimized Generated Skills to optimized_skills/ ──

OPT_SKILLS_DIR = Path("optimized_skills")
OPT_SKILLS_DIR.mkdir(exist_ok=True)

print("Saving optimized skills for all requests...")

for _, row in all_requests.iterrows():
    req_id = row["id"]
    req_text = row["request"]
    req_dir = OPT_SKILLS_DIR / f"request_{req_id}"
    req_dir.mkdir(exist_ok=True)

    try:
        result = optimized_pipeline(user_request=req_text)
        skills = result["generated_skills"]["skill_markdown_files"]
        filenames = result["generated_skills"]["file_names"]

        for content, fname in zip(skills, filenames):
            if not fname.endswith(".md"):
                fname += ".md"
            filepath = req_dir / fname
            filepath.write_text(content, encoding="utf-8")
            print(f"  [req {req_id}] Saved: {filepath}")

        # Also save the full pipeline trace as JSON
        trace_path = req_dir / "full_trace.json"
        trace_path.write_text(json.dumps(result, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

        # Log skills as MLflow artifacts
        mlflow.log_artifacts(str(req_dir), artifact_path=f"skills/optimized/request_{req_id}")

    except Exception as e:
        print(f"  [req {req_id}] ERROR: {e}")

print(f"\nOptimized skills saved to: {OPT_SKILLS_DIR.absolute()}")

# Quick summary
print("\n═══ Directory Structure ═══")
print(f"baseline_skills/       — baseline (original) skills")
print(f"optimized_skills/      — GEPA-optimized skills")
print(f"optimized_prompts/     — extracted prompt texts")


In [70]:
# ── Display Optimized vs Baseline ──

from IPython.display import display, Markdown, HTML

for agent_name in ["planner", "curator", "generator"]:
    baseline = BASELINE_PROMPTS.get(agent_name, "")
    optimized = optimized_programs.get(agent_name, "")

    display(Markdown(f"### {agent_name.title()} Agent"))

    if optimized and optimized != baseline:
        display(Markdown(
            f"**Baseline** ({len(baseline)} chars):\n<pre>{baseline}</pre>\n\n"
            f"**Optimized** ({len(optimized)} chars):\n<pre>{optimized}</pre>"
        ))
    else:
        display(Markdown(f"**No changes** or optimized prompt not extracted."))


### Planner Agent

**No changes** or optimized prompt not extracted.

### Curator Agent

**No changes** or optimized prompt not extracted.

### Generator Agent

**No changes** or optimized prompt not extracted.

## Quick Post-Optimization Check

Run the optimized pipeline on a couple training requests and compare scores.


In [71]:
# Quick validation on 2 train requests
check_ids = train_df.iloc[:2]["id"].tolist()

comparison_rows = []

for _, row in train_df.iloc[:2].iterrows():
    req_id = row["id"]
    req_text = row["request"]

    # Baseline score
    baseline_row = baseline_df[baseline_df["request_id"] == req_id]
    baseline_score = baseline_row["score"].values[0] if len(baseline_row) else None

    # Optimized score
    try:
        opt_result = optimized_pipeline(user_request=req_text)
        opt_skills = opt_result["generated_skills"]["skill_markdown_files"]
        if opt_skills:
            opt_score, opt_feedback, _ = call_judge(req_text, opt_skills[0])
        else:
            opt_score, opt_feedback = 0, "No skills"
    except Exception as e:
        opt_score, opt_feedback = 0, f"Error: {e}"

    comparison_rows.append({
        "request_id": req_id,
        "baseline_score": baseline_score,
        "optimized_score": opt_score,
        "delta": opt_score - baseline_score if baseline_score is not None else None,
    })

    print(f"Request {req_id}: baseline={baseline_score:.2f} → optimized={opt_score:.2f} (delta={opt_score - baseline_score:+.2f})")

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

avg_opt = comparison_df["optimized_score"].mean()
avg_delta = comparison_df["delta"].mean()
print(f"\nAverage optimized score: {avg_opt:.2f} (avg delta: {avg_delta:+.2f})")


2026/05/26 11:04:07 WARNING dspy.primitives.module: Failed to set LM usage. Please return `dspy.Prediction` object from dspy.Module to enable usage tracking.


Request 1: baseline=0.88 → optimized=0.88 (delta=+0.00)


2026/05/26 11:04:13 WARNING dspy.primitives.module: Failed to set LM usage. Please return `dspy.Prediction` object from dspy.Module to enable usage tracking.


Request 2: baseline=0.68 → optimized=0.80 (delta=+0.12)


,request_id,baseline_score,optimized_score,delta
0,1,0.88,0.88,0.00
1,2,0.68,0.80,0.12



Average optimized score: 0.84 (avg delta: +0.06)


[Trace(trace_id=tr-a59ddf62689074d4b8a0cb707db4017f), Trace(trace_id=tr-38f63f346007b5efedea69a37f1ab322)]

## Log to MLflow

Log the optimization results and optimized prompts as MLflow artifacts.


In [ ]:
# ── Finalize: Log Results & Artifacts to MLflow ──

with start_phase_run("summary"):
    # Log all params
    mlflow.log_params({
        "task_model": TASK_MODEL,
        "reflection_model": REFLECTION_MODEL,
        "judge_model": JUDGE_MODEL,
        "train_size": len(train_df),
        "val_size": len(val_df),
        "baseline_avg_score": avg_baseline,
    })

    # Log baseline scores table (pass DataFrame directly, not list-of-dicts)
    mlflow.log_table(baseline_df, "baseline_scores.json")

    # Log per-request baseline scores as metrics
    for _, row in baseline_df.iterrows():
        mlflow.log_metric(f"baseline_score_req_{int(row['request_id'])}", row["score"])

    # Log optimized prompts as text artifacts
    if optimized_programs:
        for name, prompt_text in optimized_programs.items():
            if prompt_text:
                mlflow.log_text(prompt_text, f"optimized_prompts/{name}_prompt.txt")

        # Also log baseline prompts for side-by-side comparison
        for name, prompt_text in BASELINE_PROMPTS.items():
            mlflow.log_text(prompt_text, f"optimized_prompts/{name}_prompt_baseline.txt")

        # Log whole prompt directory as artifacts
        mlflow.log_artifacts("optimized_prompts", artifact_path="optimized_prompts")
    else:
        print("Note: No optimized prompts extracted (GEPA may not have changed them).")

    # Log the input dataset
    if Path("skill_requests.csv").exists():
        mlflow.log_artifact("skill_requests.csv")

    print("MLflow logging complete.")
